# Lab 10a: scenario generators and their tails

**AI-powered Investment and Risk Management (25882), Lecture 10: AI in Risk Forecasting and Stress Testing**

A stress test needs scenarios: joint moves of everything you hold. This lab builds four scenario generators from history and asks the question Tail-GAN asks of a deep generator: **do the scenarios get the tails right for the portfolios you actually hold?** Every generator is fitted to 1970–2007 and then scored against what 2008–2025 delivered: the global financial crisis, COVID and the 2022 inflation shock.

**You will**

- build four generators: a historical bootstrap, a Gaussian, a multivariate Student *t* and a Gaussian mixture with regimes;
- see why the body of a distribution tells you little and the tail tells you almost everything;
- score each generator out of sample with the FZ0 loss from Lecture 6, portfolio by portfolio;
- find what none of them could produce.

**Data.** Eight US industry portfolios (banks, insurers, real estate, oil, utilities, health care, software, retail) from the Kenneth R. French Data Library, with long Treasury and corporate bond returns from the Goyal–Welch data, monthly 1970–2025, all mirrored in the teaching repository. The bond series follow Goyal–Welch's construction; their 2022 losses look closer to a 10-year than a 20-year bond.

**Runtime.** Well under a minute on a laptop.

## Setup

The helper module `airm10.py` loads the data and draws the figures. Every generator and every score is written out in this notebook. If you opened the notebook on its own, the first lines fetch the helper once; after that everything runs offline.

In [ ]:
import os
import urllib.request

if not os.path.exists("airm10.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/VitaliAlexeev/AI_Investments_2026/main/airm10.py", "airm10.py")

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import multivariate_normal, multivariate_t, norm
from sklearn.cluster import KMeans

import airm10 as A

## 1. Training months and test months

The training window ends in December 2007, so no generator sees the global financial crisis. The test window contains it, along with COVID and 2022. Change `TRAIN` and `TEST` here to move the split; everything below follows.

In [ ]:
TRAIN = ("1970-01", "2007-12")
TEST = ("2008-01", "2025-12")
N = 100_000

X = A.asset_panel()
train, test = X.loc[TRAIN[0]:TRAIN[1]], X.loc[TEST[0]:TEST[1]]
print(f"{len(train)} training months, {len(test)} test months, {X.shape[1]} assets")
display(pd.concat({"Training": A.panel_summary(train), "Test": A.panel_summary(test)}, axis=1))

Look at the last column of each block. Banks' worst month was −27.3% in the test window against −24.1% in training; corporate bonds' worst was −9.5% against −8.9%.

## 2. Four generators

Each generator turns the training months into 100,000 scenario months.

- **Bootstrap.** Draw training months at random, with replacement. Exactly as joint as history, and unable to produce a month worse than the worst one it saw.
- **Gaussian.** The multivariate normal with the training mean and covariance. Joint, smooth and thin-tailed.
- **Student *t*.** The same covariance, with fatter tails in every direction. The degrees of freedom are chosen by maximum likelihood.
- **Gaussian mixture.** Several normal distributions mixed together, with the number chosen by BIC. It is the simplest generative model that knows about regimes.

<details>
<summary><b>The maths: the four generators</b></summary>

With training months $x_1,\dots,x_T \in \mathbb{R}^{10}$, sample mean $\hat\mu$ and covariance $\hat\Sigma$:

- Bootstrap: $\tilde x = x_{U}$ with $U$ uniform on $\{1,\dots,T\}$.
- Gaussian: $\tilde x \sim N(\hat\mu, \hat\Sigma)$.
- Student *t*: $\tilde x \sim t_\nu\!\left(\hat\mu, \tfrac{\nu-2}{\nu}\hat\Sigma\right)$, which has covariance $\hat\Sigma$; $\nu$ maximises the likelihood of the training months.
- Mixture: $p(x) = \sum_{k=1}^{K} \pi_k\, N(x;\mu_k,\Sigma_k)$, fitted by expectation–maximisation; $K$ minimises $\mathrm{BIC} = -2\log \hat L + p \log T$, where $p$ counts the parameters.
</details>

In [ ]:
rng = np.random.default_rng(1)
scenarios = {"Bootstrap": train.values[rng.integers(0, len(train), N)]}

mu, S = train.mean().values, train.cov().values
scenarios["Gaussian"] = np.random.default_rng(2).multivariate_normal(mu, S, size=N)

mu_t, shape_t, df_t = A.fit_student_t(train)
scenarios[f"Student t (df {df_t})"] = multivariate_t(mu_t, shape_t, df=df_t).rvs(
    size=N, random_state=np.random.default_rng(3))
print("Student t degrees of freedom, chosen by maximum likelihood:", df_t)

In [ ]:
mixture, bic = A.fit_mixture(train)
display(bic.round(0).to_frame())
scenarios[f"Mixture ({mixture.n_components} regimes)"] = mixture.sample(N)[0]

regimes = pd.DataFrame({
    "Weight": mixture.weights_,
    "Equities, mean (% a month)": mixture.means_[:, :8].mean(axis=1),
    "Average industry volatility (% a month)": [np.sqrt(np.diag(c)[:8].mean()) for c in mixture.covariances_],
    "Treasuries, mean (% a month)": mixture.means_[:, 8],
}).round(2)
display(regimes)

BIC chose 2 regimes: a calm one with weight 0.64 and a volatile one with weight 0.36, in which industries are about 2.0 times as volatile.

## 3. Do the scenarios look real?

On the left, the whole distribution of a 60/40 portfolio's monthly return: generated months (fitted to 1970–2007) and the realised months of 2008–2025. On the right, the same curves on a log scale, which magnifies the lower tail. In the body every generator looks broadly right, if a little too wide: 1970–2007 had livelier ordinary months than 2008–2025. In the tail they separate, and in a different order. The Gaussian, too wide in the body, is too light in the tail: its 1st percentile is −7.4% against −8.8% realised. A generator that "looks realistic" has told you very little.

In [ ]:
P = A.benchmark_portfolios()
w6040 = P.loc["60/40"].values
A.cdf_figure(scenarios, test, w6040, "60/40: generated months against the months of the test window").show()

## 4. The yardstick: tails for the portfolios you hold

Tail-GAN trains its critic on exactly this: the VaR and expected shortfall of benchmark strategies. We use six portfolios: equal-weight equities, 60/40, financials (banks and insurers), banks minus utilities, credit carry (corporate bonds minus Treasuries) and a defensive mix of utilities and Treasuries.

For each generator we compute the 97.5% VaR and expected shortfall of the generated months, then score those two numbers against the realised test months with the FZ0 loss from Lecture 6. The row labelled **Realised** uses the test window's own VaR and ES: the score of a forecaster who already knew the future, and a floor for everyone else.

<details>
<summary><b>The maths: scoring a constant (VaR, ES) forecast</b></summary>

At level $\alpha = 2.5\%$, with VaR $v<0$ and expected shortfall $e<0$ (returns, so losses are negative), the FZ0 score of one realised month $y$ is

$$S(v,e;y) = -\frac{1}{\alpha e}\,\mathbf{1}\{y \le v\}\,(v-y) + \frac{v}{e} + \log(-e) - 1 .$$

VaR and ES are jointly elicitable: the expected score is minimised only by the true pair. Averaging $S$ over the test months ranks generators by how well their tail numbers describe the future. Lower is better.
</details>

In [ ]:
card = A.scorecard(scenarios, test, P)
A.es_figure(card, P, "Expected shortfall: generated from the training months, realised in the test months").show()
display(card.pivot(index="generator", columns="portfolio", values="ES").round(2))

For the 60/40 portfolio every generator is close: the realised expected shortfall was −8.8%, the bootstrap gave −8.8% and the Gaussian, the lightest as expected, −7.4%.

Credit carry and financials tell a different story. Credit carry's realised expected shortfall was −7.2%, while the generators produced between −2.7% and −3.6%. For financials the realised expected shortfall was −18.0%, against −11.2% to −12.8% generated. That is 2008: a credit and banking tail that the 1970–2007 months did not contain.

In [ ]:
display(A.fz0_table(card))

Average rank across the six portfolios, best first: Bootstrap (1.50), Mixture (2 regimes) (2.17), Student t (df 7) (2.50), Gaussian (3.83). The plain bootstrap wins because it replays the training tails exactly, and the Gaussian comes last. The mixture is close behind the bootstrap and, unlike it, can produce months worse than any it saw.

## 5. What a generator can and cannot produce

The worst single month each generator produced in 100,000 draws:

In [ ]:
display(card.pivot(index="generator", columns="portfolio", values="worst").round(1))

The bootstrap is capped at the training window's worst month: −12.8% for 60/40, while the test window delivered −13.4%. For credit carry the cap was −5.1% and reality −9.8%.

The Student *t* has no cap, but its worst draws, such as −44.6% in a single month for equal-weight equities, are not plausible either. Heavier tails are not the same as better tails, which is why plausibility has to be modelled explicitly. That is the subject of Lab 10b.

## 6. Crash breadth

A stress scenario has to be joint. Here is the share of months in which at least six of the eight industries lost more than 5%:

In [ ]:
breadth = pd.Series({"Training months": A.crash_breadth(train.values),
                     "Test months": A.crash_breadth(test.values),
                     **{name: A.crash_breadth(Z) for name, Z in scenarios.items()}})
display((100 * breadth).round(1).to_frame("Months with at least 6 of 8 industries down more than 5% (%)"))

In training it was 5.5% of months; in the test window 7.9%. The generators reproduce the past (4.4% to 5.5%), not the future: 2008–2025 was more joint than 1970–2007.

## What to notice

1. **The body says little; the tail decides.** All four generators get a 60/40 portfolio's typical month roughly right; their tails differ far more, and not in the same order.
2. **The yardstick is portfolio-specific.** Generators that are fine for 60/40 understate the credit tail of 2008 by half or more, and the banking tail by about a third.
3. **Every generator inherits its history.** The bootstrap cannot go beyond it; the Student *t* goes beyond it in every direction at once; the mixture adds a volatile regime but not 2008's credit channel.

## Exercises

**Core 1.** Move the split: `TRAIN = ("1970-01", "2019-12")` and `TEST = ("2020-01", "2025-12")`, then rerun. Which generator wins now, and does any reproduce 2022 for the 60/40 portfolio?

**Core 2.** Add a portfolio of your own to `P`, for example long software and short utilities, and rerun the scorecard. Is the ranking of generators the same for every portfolio?

**Stretch 1.** Tail-GAN's idea in miniature: choose the mixture's number of regimes by the average FZ0 score on a validation window (hold out 2000–2007 from training) instead of by BIC. Does the choice change?

**Stretch 2.** Add any generator you like (a GAN, a variational autoencoder, a time-series foundation model) to the `scenarios` dictionary. The scorecard does not care how scenarios are made; that is the point of the yardstick.